# Kvartalsvis pro forma-resultaträkning med PROC COMPUTAB

## Sammanfattning

Denna notebook bygger en regionalbanks kvartalsvisa pro forma-resultaträkning direkt från månatlig huvudboksdata med PROC COMPUTAB, tabellrapportproceduren i SAS/ETS. Den dirigerar varje månads ränteintäkter, räntekostnader, avgiftsintäkter och driftskostnader till rätt kalenderkvartalskolumn, och använder sedan programmerbara radblock för att beräkna räntenetto, resultat före skatt och nettoresultat, samt ett kolumnblock för att rulla ihop de fyra kvartalen till en helårssumma.

## Datakällor

Det enda syntetiska datasetet `bank_ledger` simulerar ett räkenskapsår av månatliga poster i finansiella rapporter för en medelstor lokalbank. Tolv månatliga observationer genereras direkt i koden med `call streaminit`/`rand` så att notebooken är helt fristående.

| Variabel | Typ | Beskrivning |
|----------|------|-------------|
| `stmtdate` | Num (DATE9.) | Månadsslutets rapportdatum (jan–dec) |
| `loanint`  | Num | Ränta och avgifter intjänade på låneportföljen (tusental USD) |
| `secint`   | Num | Ränta intjänad på placeringsvärdepappersboken (tusental USD) |
| `depint`   | Num | Ränta betald på inlåning (tusental USD) |
| `borrint`  | Num | Ränta betald på upplåning / FHLB-förskott (tusental USD) |
| `feeinc`   | Num | Icke-ränteintäkter (avgifts- och serviceintäkter) (tusental USD) |
| `salaries` | Num | Löner och personalförmånskostnader (tusental USD) |
| `occupancy`| Num | Lokal- och utrustningskostnad (tusental USD) |
| `othropex` | Num | Övriga icke-ränterelaterade driftskostnader (tusental USD) |
| `provision`| Num | Avsättning för kreditförluster (tusental USD) |
| `taxrate`  | Num | Effektiv skattesats tillämpad på resultat före skatt |

# Kvartalsvis pro forma-resultaträkning med PROC COMPUTAB

Bankers finansteam förvandlar rutinmässigt en månatlig huvudbok till en **kvartalsvis resultaträkning** som visar räntenetto och nettoresultat på sista raden. `PROC COMPUTAB` (SAS/ETS) är byggd för just detta: den lägger ut en programmerbar tabell vars **kolumner** är rapportperioderna och vars **rader** är rapportens radposter, och den låter dig beräkna delsummor med vanlig DATA-stegslogik i rad- och kolumnblock.

I denna notebook:

1. Genererar vi ett räkenskapsår av syntetisk månatlig huvudboksdata för en lokalbank.
2. Dirigerar vi varje månad till dess kalenderkvartal och bygger en kvartalsvis resultaträkning.
3. Beräknar vi räntenetto, resultat före skatt och nettoresultat i ett **radblock**, och rullar ihop kvartalen till en helårssumma i ett **kolumnblock**.
4. Återanvänder vi `out=`-tabellen så att den beräknade rapporten kan mata efterföljande rapportering.

## Steg 1 — Generera syntetisk månatlig huvudboksdata

Vi simulerar tolv observationer vid månadsslut. Ränteintäkter från lån och värdepapper trendar svagt uppåt över året, kostnaderna för inlåning och upplåning skalar med ränteläget, och avgiftsintäkter, driftskostnader och kreditförlustavsättningen bär realistiskt brus från månad till månad. `call streaminit` fixerar fröet så att data är reproducerbara.

In [1]:
data bank_ledger;
   CALL streaminit(20240115);
   format stmtdate date9.;
   GÖR month = 1 TILL 12;
      /* Rapportdatum vid månadsslut för räkenskapsåret 2024 */
      stmtdate = mdy(month, 28, 2024);

      /* Svag uppåtgående drift över året + månatligt brus */
      drift = 1 + 0.012 * (month - 1);

      /* Ränteintäkter (tusental USD) */
      loanint = round(1850 * drift + rand('normal', 0, 45), 0.01);
      secint  = round( 420 * drift + rand('normal', 0, 15), 0.01);

      /* Räntekostnader (tusental USD) */
      depint  = round( 540 * drift + rand('normal', 0, 22), 0.01);
      borrint = round( 130 * drift + rand('normal', 0, 10), 0.01);

      /* Icke-ränteintäkter och -kostnader (tusental USD) */
      feeinc    = round(310 + rand('normal', 0, 18), 0.01);
      salaries  = round(720 + rand('normal', 0, 25), 0.01);
      occupancy = round(165 + rand('normal', 0, 8),  0.01);
      othropex  = round(240 + rand('normal', 0, 20), 0.01);

      /* Avsättning för kreditförluster, ibland förhöjd */
      provision = round(95 + rand('exponential') * 40, 0.01);

      /* Effektiv skattesats */
      taxrate = 0.21;

      UTDATA;
   SLUT;
   BEHÅLL stmtdate loanint secint depint borrint
        feeinc salaries occupancy othropex provision taxrate;
KÖR;

PROCEDUR SKRIV data=bank_ledger noobs ETIKETT;
   TITEL 'Syntetisk månatlig huvudbok — lokalbank RÅ2024';
   ETIKETT stmtdate="Rapportdatum"
           loanint="Ränta & avgifter på lån"
           secint="Ränta på värdepapper"
           depint="Ränta på inlåning"
           borrint="Ränta på upplåning"
           feeinc="Avgiftsintäkter"
           salaries="Löner & förmåner"
           occupancy="Lokaler & utrustning"
           othropex="Övriga driftskostnader"
           provision="Kreditförlustavsättning"
           taxrate="Skattesats";
   format loanint secint depint borrint feeinc
          salaries occupancy othropex provision comma8.2;
KÖR;

                                     Syntetisk månatlig huvudbok — lokalbank RÅ2024                                     

Rapportdatum     Ränta & avgifter på lån     Ränta på värdepapper     Ränta på inlåning     Ränta på upplåning   Avgiftsintäkter     Löner & förmåner  Lokaler & utrustning   Övriga driftskostnader    Kreditförlustavsättning  Skattesats
   28JAN2024                    1,772.95                   423.79                531.47                 128.99            339.33               699.38                171.95                   202.43                     130.93        0.21
   28FEB2024                    1,824.38                   421.13                564.85                 125.53            294.09               767.29                162.79                   307.61                     123.25        0.21
   28MAR2024                    1,931.98                   442.06                536.64                 131.71            295.72               724.03                153.2


NOTE: DATA bank_ledger


NOTE: Wrote bank_ledger (12 rows, 11 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=bank_ledger

NOTE: PROC PRINT completed: 12 observations printed, 11 variables


## Steg 2 — Bygg den kvartalsvisa resultaträkningen

Hjärtat i rapporten är `PROC COMPUTAB`-steget nedan.

- **`columns qtr1 qtr2 qtr3 qtr4 fy;`** definierar fyra kvartalskolumner plus en helårskolumn.
- Det omärkta **indatablocket** sätter den automatiska variabeln **`_col_`** med `qtr(stmtdate)`, vilket dirigerar varje månatlig observation till rätt kvartalskolumn. Eftersom indata transponeras som standard hamnar varje huvudboksvariabel på den rad som delar dess namn.
- **Radblocket** `inc_stmt:` körs en gång per kolumn och beräknar de härledda raderna — räntenetto, totala icke-ränterelaterade kostnader, resultat före skatt och nettoresultat — precis som en redovisare skulle göra.
- **Kolumnblocket** `total:` körs en gång per rad och summerar de fyra kvartalen till `fy`-kolumnen (helår).

Satserna `rows ... / ...` lägger till titlar, indrag och linjeregler (`ol` överlinje, `ul` underlinje, `dul` dubbel underlinje) så att utdata läses som en riktig resultaträkning.

In [2]:
TITEL 'Pro forma-resultaträkning — Community Bancorp, Inc.';
title2 'Räkenskapsår 2024  (belopp i tusental USD)';

PROCEDUR computab data=bank_ledger cspace=2 cwidth=11 out=qtr_income;

   /* Fyra kvartal plus en ihoprullad helårskolumn */
   columns qtr1 qtr2 qtr3 qtr4 fy / format=comma11.1;
   columns qtr1 / 'K1';
   columns qtr2 / 'K2';
   columns qtr3 / 'K3';
   columns qtr4 / 'K4';
   columns fy   / 'Hela' 'året' +3;

   /* Resultaträkningens rader, uppifrån och ner */
   rows loanint  / 'Ränta & avgifter på lån';
   rows secint   / 'Ränta på värdepapper'          ul;
   rows intinc   / 'Summa ränteintäkter';
   rows depint   / 'Ränta på inlåning';
   rows borrint  / 'Ränta på upplåning'            ul;
   rows intexp   / 'Summa räntekostnader';
   rows nii      / 'Räntenetto'                     ol skip;
   rows provision/ 'Avsättning för kreditförluster' ul;
   rows niiap    / 'Räntenetto efter avsättning'    skip;
   rows feeinc   / 'Icke-ränteintäkter'             ul;
   rows salaries / 'Löner & förmåner';
   rows occupancy/ 'Lokaler & utrustning';
   rows othropex / 'Övriga driftskostnader'         ul;
   rows nonintexp/ 'Summa icke-räntekostnader'      skip;
   rows pretax   / 'Resultat före skatt'            ol;
   rows taxexp   / 'Skattekostnad'                  ul;
   rows netinc   / 'Nettoresultat'                  dul;

   /* Indatablock: dirigera varje månad till dess kalenderkvartal */
   _col_ = qtr(stmtdate);

   /* Radblock: beräkna delsummor för rapporten över varje kolumn */
   inc_stmt:
      intinc    = loanint + secint;
      intexp    = depint + borrint;
      nii       = intinc - intexp;
      niiap     = nii - provision;
      nonintexp = salaries + occupancy + othropex;
      pretax    = niiap + feeinc - nonintexp;
      taxexp    = pretax * 0.21;
      netinc    = pretax - taxexp;

   /* Kolumnblock: rulla ihop de fyra kvartalen till helårskolumnen */
   total:
      fy = qtr1 + qtr2 + qtr3 + qtr4;
KÖR;

                                  Pro forma-resultaträkning — Community Bancorp, Inc.                                   
                                       Räkenskapsår 2024  (belopp i tusental USD)                                       


                             The COMPUTAB Procedure                             

                             K1           K2           K3           K4         Hela  
                                                                               året  
                           qtr1         qtr2         qtr3         qtr4           fy  
                    -----------  -----------  -----------  -----------  -----------  
  Ränta & avgifter på lån
  loanint               5,529.3      5,818.7      5,963.5      6,277.0     23,588.4  
  Ränta på värdepapper
  secint                1,287.0      1,334.9      1,342.0      1,452.9      5,416.8  
                    -----------  -----------  -----------  -----------  -----------  
  Summa ränteintäkter
 


NOTE: Option TITLE changed to Pro forma-resultaträkning — Community Bancorp, Inc..
NOTE: Option TITLE2 changed to Räkenskapsår 2024  (belopp i tusental USD).
NOTE: PROC COMPUTAB
NOTE: COMPUTAB OUT= dataset written to: qtr_income.csv
NOTE: PROC COMPUTAB statement used.


## Steg 3 — Återanvänd COMPUTAB-utdatadatasetet

`PROC COMPUTAB`-steget ovan skrev sin beräknade tabell till `qtr_income` via `out=`. Varje rad i det datasetet är en radpost i rapporten och varje kolumnvariabel (`qtr1`–`qtr4`, `fy`) håller det beräknade värdet, så det kan mata efterföljande rapportering. Nedan skriver vi ut den ihoprullade helårskolumnen för att bekräfta att siffrorna stämmer.

In [3]:
TITEL 'COMPUTAB-utdatadataset — helårskolumn';

PROCEDUR SKRIV data=qtr_income ETIKETT noobs;
   VARIABEL _row_ fy;
   format fy comma12.1;
   ETIKETT _row_ = 'Radpost' fy = 'Helår';
KÖR;

TITEL;

                                         COMPUTAB-utdatadataset — helårskolumn                                          
                                       Räkenskapsår 2024  (belopp i tusental USD)                                       


  Radpost     Helår
---------  --------
loanint    23,588.4
secint      5,416.8
intinc     29,005.2
depint      6,831.2
borrint     1,650.7
intexp      8,482.0
nii        20,523.2
provision   1,568.9
niiap      18,954.3
feeinc      3,703.2
salaries    8,763.1
occupancy   1,985.6
othropex    2,944.8
nonintexp  13,693.5
pretax      8,964.1
taxexp      1,882.5
netinc      7,081.6




NOTE: Option TITLE changed to COMPUTAB-utdatadataset — helårskolumn.
NOTE: PROC PRINT data=qtr_income

NOTE: PROC PRINT completed: 17 observations printed, 2 variables


## Tolka resultaten

Pro forma-rapporten läses uppifrån och ner som en banks tillsynsmässiga call report: totala ränteintäkter minus totala räntekostnader ger **räntenetto** — här omkring \$20,5 miljoner för året — bankens främsta intjäningsdrivkraft. Att subtrahera **avsättningen för kreditförluster**, lägga till **avgiftsintäkter** och räkna av **icke-ränterelaterade kostnader** ger ett resultat före skatt på ungefär \$9,0 miljoner, och att tillämpa den effektiva skattesatsen på 21 % ger ett **nettoresultat** nära \$7,1 miljoner. Kolumnblocket `total:` adderar de fyra kvartalen till helårskolumnen, så årssummorna stämmer med summan av kvartalen per konstruktion — bekräftat i `out=`-datasetet, där helårets `netinc` på 7 081,6 är lika med summan av de fyra kvartalssiffrorna.

Eftersom allt beräknas inne i `PROC COMPUTAB`:s programmerbara rad- och kolumnblock kräver ett byte till en verklig månatlig huvudbok ingen ändring av rapportlogiken — endast indatadatasetet ändras. `out=`-datasetet gör sedan den beräknade rapporten tillgänglig för instrumentpaneler, trendanalys eller automatisering av styrelseunderlag.